<div class="alert alert-block alert-warning">
<b>版权声明</b>\n\n本课件版权归课程负责人所有。未经书面许可，严禁以任何形式复制、转载、传播或用于商业用途。\n\n© 2026 深度学习讲习班 版权所有\n</div>

In [ ]:
import numpy as np
import random
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# <font color="#00FFFF" > 第二章 神经网络训练流程

## 本章内容
> 1. **梯度下降法及其可视化**
> 2. **模型训练流程的优化**
> 2. **自动微分和计算图**
> 3. **构建神经网络训练代码**


##  <font color="#FFEA00" >第一节 梯度下降法

### **<font color="#39FF14" size=6 >核心概念清单</font>**

#### 1. 梯度下降法的定义

- [ ] <font color="#FF00A0">**Gradient descent**</font> 的基本定义
是一种无约束数学优化的方法，是一种一阶迭代优化算法，其用于寻找可微函数的局部最小值
是一种迭代技术，试图从初始猜测开始，为给定模型，数据点，和损失函数找到的最佳可能的参数
- [ ] 无约束数学优化 vs 有约束数学优化（SVM）
- [ ] 一阶迭代优化算法（二阶有牛顿法）
- [ ] 可微多元函数的局部最小值


---

#### 2. 损失函数（Loss Function）

- [ ] 单个样本误差：$\text{error}_i = \hat{y}_i - y_i$
- [ ] 代价函数（Cost Function）的定义
- [ ] <font color="#FF00A0">**Loss Surface**</font> 的概念

---

#### 3. 梯度下降的类型

| 类型 | 数据点数量 | 特点 | 适用场景 |
|------|-----------|------|---------|
| <font color="#FF00A0">**批量梯度下降**</font> | n=N | 收敛稳定，计算量大 | 小数据集 |
| <font color="#FF00A0">**随机梯度下降**</font> | n=1 | 噪声大，收敛快 | 在线学习 |
| <font color="#FF00A0">**小批量梯度下降**</font> | 1<n<N | 平衡稳定与效率 | 大数据集（最常用） |


---

#### 4. 学习率（Learning Rate）

- [ ] 学习率作为超参数的概念
- [ ] 学习率过小的影响：收敛缓慢
- [ ] 学习率过大的影响：震荡或发散
- [ ] 最优学习率的选择策略


---

#### 5. Epoch的概念

- [ ] <font color="#FF00A0">**Epoch**</font> 的定义：训练集中的所有数据点都完成一次前向传播和反向传播
- [ ] Epoch vs Batch vs Iteration 的区别
- [ ] 不同梯度下降类型下，一个Epoch内的参数更新次数

### **<font color="#39FF14" size=6 >回忆：机器学习中的梯度下降法</font>**

### **<font color="#39FF14" size=6 >梯度下降的定义</font>**

- <font color="#FF00A0">**Gradient descent**</font> is a method for unconstrained <font color="#FF00A0">**mathematical optimization**</font>. It is a <font color="#FF00A0">**first-order iterative algorithm**</font> for minimizing a <font color="#FF00A0">**differentiable multivariate function**</font>.
- 梯度下降法是一种无约束数学优化的方法，是一种一阶迭代优化算法，其用于寻找可微函数的局部最小值。

### **<font color="#39FF14" size=6 >深入理解梯度下降</font>**

#### <font color="#CCFF00">**模型**</font>：$y = b + wx$

#### <font color="#CCFF00">**合成数据生成**</font>

In [ ]:
true_b = 1
true_w = 2
N = 100

# Data Generation
np.random.seed(42)  # 不要忘了设置随机种子
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_w * x + epsilon

In [ ]:
# Shuffles the indices
idx = np.arange(N)
np.random.shuffle(idx)

# Uses first 80 random indices for train
train_idx = idx[:80]
# Uses the remaining indices for validation
val_idx = idx[80:]

# Generates train and validation sets
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

#### <font color="#CCFF00">**Step 0: 随机初始化参数**</font>

In [ ]:
# Step 0 - Initializes parameters "b" and "w" randomly
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)

print(b, w)

#### <font color="#CCFF00">**Step 1: 计算模型的预测值——前向传播**</font>

In [ ]:
# Step 1 - Computes our model's predicted output - forward pass
yhat = b + w * x_train

#### <font color="#CCFF00">**Step 2: 计算损失**</font>

计算代价函数Cost Function（一组数据点的误差和），我们可以采用什么策略呢？
- loss function是一个数据的误差
- 计算所有所有数据点(n=N)的损失——<font color="#FF00A0">**批量梯度下降**</font>
- 计算一个数据点(n=1)的损失——<font color="#FF00A0">**随机梯度下降**</font>
- 计算介于1和N之间的n个数据点的损失——<font color="#FF00A0">**小批量梯度下降**</font>

接下来，我们使用批量梯度下降来计算损失，n=N=80
- 对于回归问题，通常使用MSE计算损失，平方以后，损失会放大，除以之后和数据量关系不大，
- mse和极大似然估计是等价的
- loss surface（两个参数就是一个三位曲线图了）

In [ ]:
# Step 2 - Computing the loss
error = (yhat - y_train)

loss = (error ** 2).mean()
print(loss)

#### <font color="#CCFF00">**Step 3: 计算梯度**</font>
mse的推导最后可以得到b和w的梯度大小
- 谁梯度大谁的变化更大，更新的就越快



In [ ]:
# Step 3 - Computes gradients for both "b" and "w" parameters
b_grad = 2 * error.mean()
w_grad = 2 * (error * x_train).mean()
print(b_grad, w_grad)

#### <font color="#CCFF00">**Step 4: 更新参数**</font>
- 使用梯度去更新参数，需要考虑超参数：学习率
- 超参数概念？
- 如何选择学习率
- 参数更新公式：数据减去学习率乘以梯度得到新数据

In [ ]:
lr = 0.1
print(b, w)

# Step 4 - Updates parameters using gradients and the
# learning rate
b = b - lr * b_grad
w = w - lr * w_grad
print(b, w)

#### <font color="#CCFF00">**Step 5: 循环往复**</font>

- 由前面的步骤1-4，完成了一次参数更新；
- 使用更新后的参数返回步骤1，重启1-4的步骤。

<font color="#FF00A0">**Epoch**</font>的定义：（代次）
- 训练集(N)中的N个数据点都已经用于所有的步骤：前向传播、计算损失、计算梯度、更新参数，则一个周期完成。

使用的梯度下降类型将决定一个Epoch内参数更新的次数：

- 批量梯度下降(n=N)，一个Epoch更新_1_次参数；
- 随机梯度下降(n=1)，一个Epoch更新_N_次参数；
- 小批量梯度下降(n)，一个Epoch更新_N/n_次参数。

- 为什么同样的100个epoch之后，mini更快逼近  --因为更新参数更快，更接近 尽管稳定性不是太好
- 同样epoch后，随机梯度下降在初始逼近极快，但是之后很难稳定趋向极值  更新很快，稳定性最差，还有偏差

##  <font color="#FFEA00" >第二节 模型训练流程的优化

### **<font color="#39FF14" size=6 >核心概念清单</font>**

#### 1. 使用PyTorch进行模型训练的基本步骤

- [ ] 数据准备：从Numpy到PyTorch Tensor的转换
- [ ] 模型配置：模型、损失函数、优化器
- [ ] 训练循环：前向传播 → 计算损失 → 反向传播 → 更新参数

---

#### 2. 张量（Tensor）的概念

- [ ] <font color="#FF00A0">**张量（Tensor）**</font> 的定义：深度学习特化版的多维数组
- [ ] 张量 vs NumPy ndarray 的区别
- [ ] 张量的核心特性：
  - 记录计算图（Computational Graph）
  - 自动微分（Autograd）：`.backward()` 方法
  - GPU加速：`.to(device)` 方法

**重要提示**：
- 注意张量操作是否共享底层数据
- 区分原位操作（in-place）和非原位操作
- 深度学习通常处理4维张量（Batch, Channel, Height, Width）

---

#### 3. 计算设备（CPU/GPU/MPS）

- [ ] 设备选择代码：`device = 'cuda' if torch.cuda.is_available() else 'mps'`
- [ ] 为什么深度学习需要GPU？
  - GPU的大规模并行计算能力
  - CPU vs GPU的架构差异
- [ ] 数据流：CPU加载数据 → 预处理 → 发送到GPU → 模型训练

---

#### 4. 随机种子的设置

- [ ] 为什么要设置随机种子？—— 保证实验可复现性
- [ ] 需要设置的多处随机种子：
  ```python
  torch.backends.cudnn.deterministic = True  # 强制确定性算法
  torch.backends.cudnn.benchmark = False     # 关闭自动调优
  torch.manual_seed(seed)                    # PyTorch随机种子
  np.random.seed(seed)                       # NumPy随机种子
  random.seed(seed)                          # Python random种子
  ```

⚠️ **注意**：即使设置了相同种子，硬件/环境/cuda版本变化仍可能导致结果差异

---

#### 5. 自动微分（Autograd）

- [ ] PyTorch自动计算梯度的机制
- [ ] `requires_grad=True` 的作用
- [ ] `.backward()` 方法的使用

---

#### 6. 梯度的累加与清零

- [ ] 梯度累加特性：每次 `.backward()` 会累加到 `.grad`
- [ ] `.grad.zero_()` 的必要性
- [ ] `torch.no_grad()` 上下文管理器的作用
- [ ] 叶子张量（leaf tensor）vs 非叶子张量的区别

**关键代码模式**：
```python
loss.backward()      # 计算梯度
optimizer.step()     # 更新参数
optimizer.zero_grad() # 清零梯度（重要！）
```

In [ ]:
# Step 0 - Initializes parameters "b" and "w" randomly
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)

print(b, w)

# Sets learning rate - this is "eta" ~ the "n"-like Greek letter
lr = 0.1
# Defines number of epochs
n_epochs = 1000

# 请完成包含n_epochs次训练循环的代码
for epoch in range(n_epochs):
    yhat = b + w * x_train
    error = (yhat - y_train)
    loss = (error ** 2).mean()
    b_grad = 2 * error.mean()
    w_grad = 2 * (error * x_train).mean()
    b = b - lr * b_grad
    w = w - lr * w_grad

print(b, w)

- ①Step 0: Random initialization of parameters / weights
- ②Initialization of hyper-parameters
- ③<font color="#FF00A0" >**Step 1**</font>: Forward pass
- ④<font color="#FF00A0" >**Step 2**</font>: Computing loss
- ⑤<font color="#FF00A0" >**Step 3**</font>: Computing gradients
- ⑥<font color="#FF00A0" >**Step 4**</font>: Updating parameters
- Repeat Step 1-4

In [ ]:
# Sanity Check: do we get the same results as our
# gradient descent?
linr = LinearRegression()
linr.fit(x_train, y_train)
print(linr.intercept_, linr.coef_[0])

### **<font color="#39FF14" size=6 >使用PyTorch进行模型训练</font>**

#### <font color="#CCFF00">**设置计算设备**</font>

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torchviz import make_dot

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # or 'mps'
print(device)

In [ ]:
n_cudas = torch.cuda.device_count()
for i in range(n_cudas):
    print(torch.cuda.get_device_name(i))

#### <font color="#CCFF00">**深度学习中为什么要使用GPU**</font>
- 张量 维度 多维数组
- 张量是容易转移到gpu上面，并且容易计算的内容，并且可以通过.backward（）自动计算梯度，他是支持自动微分和gpu加速的多维数组，

#### <font color="#CCFF00">**将训练数据存入设备**</font>

In [ ]:
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

print(type(x_train), type(x_train_tensor), x_train_tensor.type())

#### <font color="#CCFF00">**设置随机种子及初始化**</font>

In [ ]:
# RECOMMENDED!
# Step 0 - Initializes parameters "b" and "w" randomly
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
print(b, w)

In [ ]:
def set_seed(self, seed=42):
    torch.backends.cudnn.deterministic = True # 强制 cuDNN 只使用确定性的算法
    torch.backends.cudnn.benchmark = False # 关闭 cuDNN 的自动调优功能
    torch.manual_seed(seed) # 固定 PyTorch 的随机数生成器(CPU和GPU权重初始化、Dropout等）
    np.random.seed(seed) # 固定 NumPy 库的随机数生成器
    random.seed(seed) # 固定标准 random 模块的随机数生成器
    try:
        self.train_loader.sampler.generator.manual_seed(seed) # 固定DataLoader采样的随机种子
    except AttributeError:
        pass

#### <font color="#CCFF00">**梯度的累加和清零**</font>

In [ ]:
# Step 1
yhat = b + w * x_train_tensor

# Step 2
error = (yhat - y_train_tensor)
loss = (error ** 2).mean()

# Step 3
loss.backward()

In [ ]:
print(b.grad, w.grad)

In [ ]:
b.grad.zero_(), w.grad.zero_()

#### <font color="#CCFF00">**训练流程的优化：自动微分**</font>

In [ ]:
# Step 0 - Initializes parameters "b" and "w" randomly
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)

lr = 0.1
n_epochs = 1000

for epoch in range(n_epochs):
    # Step 1
    yhat = b + w * x_train_tensor

    # Step 2
    error = (yhat - y_train_tensor)
    loss = (error ** 2).mean()

    # Step 3
    loss.backward()

    # Step 4 - Updates parameters
    # 参数更新代码尝试
    with torch.no_grad():
        b -= lr * b.grad
        w -= lr * w.grad

    b.grad.zero_()
    w.grad.zero_()

print(b, w)

##  <font color="#FFEA00" >第三节 自动微分和计算图

### **<font color="#39FF14" size=6 >核心概念清单</font>**

#### 1. 反向传播的核心目的

- [ ] 神经网络训练本质：找到使损失函数最小的权重
- [ ] 梯度指向损失函数上升最快的方向
- [ ] 沿梯度反方向更新权重（梯度下降）

---

#### 2. 链式法则（Chain Rule）

- [ ] 链式法则的数学表达
- [ ] 反向传播 = 多变量复合函数的链式法则在计算机程序中的应用
- [ ] 局部敏感度的概念

---

#### 3. 前向传播 vs 反向传播

| 阶段 | 方向 | 计算内容 | 目的 |
|------|------|---------|------|
| **前向传播** | 输入 → 输出 | 计算预测值和损失 | 得到损失值 |
| **反向传播** | 输出 → 输入 | 计算梯度 | 得到参数更新方向 |

**两阶段流程**：
1. 前向传播：构建计算图，计算损失
2. 反向传播：沿计算图反向传播梯度

---

#### 4. 计算图（Computational Graph）

- [ ] <font color="#FF00A0">**计算图**</font> 的定义：描述数学运算的有向无环图（DAG）
- [ ] 计算图的组成元素：
  - 节点：变量（蓝色框 = 需要梯度的参数）
  - 边：运算操作（灰色框）
  - 叶子节点：计算起点（绿色框）

**注意**：计算图只包含参与梯度计算的张量

---

#### 5. 动态计算图（Define-by-Run）

- [ ] <font color="#FF00A0">**动态计算图**</font> 的定义：每次前向传播时实时构建计算图
- [ ] PyTorch的动态计算图特点：
  - 每次前向传播可以不同
  - 支持条件分支、循环等动态结构
  - 内存效率高（用完即释放）
- [ ] 动态计算图 vs 静态计算图（TensorFlow 1.x）

**PyTorch优势**：灵活、易于调试、支持动态网络结构

### **<font color="#39FF14" size=6 >反向传播的核心</font>**

#### <font color="#CCFF00">**核心目的：求梯度（求导）**</font>

- 神经网络的训练本质是优化问题：找到一组权重（Weights），使得损失函数（Loss）最小。
- 梯度指向损失函数上升最快的方向，因此我们沿着梯度的反方向更新权重（梯度下降）。

#### <font color="#CCFF00">**数学基础：链式法则（Chain Rule）**</font>

#### <font color="#CCFF00">**两个阶段：前向传播 vs 反向传播**</font>

### <font color="#FF6B00" size=6 >**作业3:深度学习框架的自动微分实现**</font>

> **参考资料**：`/Users/pandamas/work/03深度学习讲习班/参考资料/Build Framework/part01.ipynb`

**本作业旨在通过阅读简化版自动微分框架的实现代码，深入理解深度学习框架中自动微分的核心原理。**
- **作业要求**：

请仔细阅读参考资料 notebook，按顺序理解以下核心内容：

| 步骤 | 核心概念 | 需要理解的关键点 |
|------|---------|------------------|
| 步骤1-3 | Variable & Function | 变量作为"箱子"、函数封装、正向传播流程 |
| 步骤4 | 数值微分 | 中心差分近似的原理、数值微分的两个问题（精度丢失、计算成本高） |
| 步骤5 | 反向传播理论 | 链式法则、计算图结构、局部梯度的概念 |
| 步骤6-7 | 手动→自动反向传播 | `grad` 属性、`creator` 机制、动态计算图的构建原理 |
| 步骤8 | 递归→循环 | 为什么将递归改为循环？循环实现的优势是什么？ |
| 步骤9 | 易用性改进 | 类型检查、`as_array` 的作用、梯度自动初始化 |
| 步骤10 | 梯度检验 | 使用数值微分验证反向传播正确性的方法 |

**阅读建议**：
- 建议边读边动手运行代码，观察中间结果
- 重点关注 `Variable` 类和 `Function` 类的演变过程
- 理解 `creator` 属性如何建立变量与函数之间的连接

**阅读后请思考如下问题**：

1. 为什么要把数据包装成 `Variable` 类，而不是直接使用 numpy 数组？这样设计的好处是什么？
2. `Variable` 类中的 `data` 和 `grad` 分别代表什么？它们之间是什么关系？
3. 解释什么是**动态计算图**（Define-by-Run）。与静态计算图相比，它有什么特点？
4. `creator` 属性的作用是什么？它是在什么时候被设置的？请描述正向传播时如何建立变量与函数之间的"连接"。
5. 在代码中，链式法则是如何体现的？请指出具体代码位置并解释。
6. 参考资料中步骤7使用**递归**实现 `backward`，步骤8改为使用**循环**实现，循环实现相比递归实现有什么优势？请从内存使用和执行效率角度分析。
7. 请结合参考资料解释，为什么训练模型（Training）比模型推理模式（Inference）要消耗大得多显存（GPU Memory），请指出具体代码位置并解释。
8. 前向模式自动微分（Forward-mode AD）和反向模式自动微分（Reverse-mode AD）各自的原理是什么？为什么在深度学习中普遍采用反向模式？（提示：考虑输入维度和输出维度的差异）

- 作业不需要提交，自行学习理解即可。

### **<font color="#39FF14" size=6 >计算图</font>**

In [ ]:
# Step 0 - Initializes parameters "b" and "w" randomly
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

# Step 1 - Computes our model's predicted output - forward pass
yhat = b + w * x_train_tensor

# Step 2 - Computes the loss
# We are using ALL data points, so this is BATCH gradient
# descent. How wrong is our model? That's the error!
error = (yhat - y_train_tensor)
# It is a regression, so it computes mean squared error (MSE)
loss = (error ** 2).mean()

# We can try plotting the graph for any python variable:
# yhat, error, loss...
make_dot(yhat)

In [ ]:
b_nograd = torch.randn(1, requires_grad=False, \
                       dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

yhat = b_nograd + w * x_train_tensor

make_dot(yhat)

##  <font color="#FFEA00" >第四节 构建神经网络训练代码

### **<font color="#39FF14" size=6 >核心概念清单</font>**

#### 1. 优化器（Optimizer）

- [ ] <font color="#FF00A0">**优化器**</font> 的定义：用于更新和调整神经网络权重的算法
- [ ] 优化器的作用：计算梯度 → 引导参数向损失减小方向更新
- [ ] 优化器的使用模式：`optimizer.step()` + `optimizer.zero_grad()`

---

#### 2. 常见优化算法

| 优化器 | 特点 | 适用场景 |
|--------|------|---------|
| <font color="#FF00A0">**SGD**</font> | 随机梯度下降，简单稳定 | 基础训练 |
| <font color="#FF00A0">**Adam**</font> | 自适应学习率，收敛快 | 大多数场景（最常用） |
| AdamW | 带权重衰减的Adam | 需要正则化时 |
| RMSprop | 适合非平稳目标 | RNN训练 |

- [ ] 后续会详细介绍各种优化器及其算法的设计

**创建优化器**：`optimizer = optim.SGD(model.parameters(), lr=0.1)`

---

#### 3. 损失函数

- [ ] PyTorch内置损失函数：`nn.MSELoss()`, `nn.CrossEntropyLoss()` 等
- [ ] 损失函数的参数：`reduction='mean'` 或 `'sum'`

**创建损失函数**：`loss_fn = nn.MSELoss(reduction='mean')`

---

#### 4. 模型定义（nn.Module）

- [ ] 继承 `nn.Module` 类定义模型
- [ ] `__init__()` 方法：定义模型结构和参数
- [ ] `forward()` 方法：定义前向传播逻辑
- [ ] `model.state_dict()`：获取模型状态（用于保存/加载）


---

#### 5. 参数初始化

- [ ] PyTorch默认初始化方法
- [ ] 常用初始化策略：Xavier初始化、He初始化
- [ ] 后续会详细讲解各种初始化策略

---

#### 6. model.train() 和 model.eval()

| 方法 | 作用 | 影响 | 使用场景 |
|------|------|------|---------|
| `model.train()` | 设置训练模式 | Dropout启用、BatchNorm使用batch统计 | 训练阶段 |
| `model.eval()` | 设置评估模式 | Dropout关闭、BatchNorm使用running统计 | 验证/测试/部署阶段 |

- [ ] 后续会详细讲解Dropout以及BatchNorm的设计逻辑及作用

⚠️ **重要**：忘记设置正确模式会导致预测结果不一致！

---

#### 7. 完整训练流程框架


### **<font color="#39FF14" size=6 >使用优化器更新参数</font>**

In [ ]:
lr = 0.1

# Step 0 - Initializes parameters "b" and "w" randomly
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

# Defines an SGD optimizer to update the parameters
optimizer = optim.SGD([b, w], lr=lr)

# Defines number of epochs
n_epochs = 1000

for epoch in range(n_epochs):
    # Step 1
    yhat = b + w * x_train_tensor

    # Step 2
    error = (yhat - y_train_tensor)
    loss = (error ** 2).mean()

    # Step 3
    loss.backward()

    # Step 4
    optimizer.step()

    optimizer.zero_grad()

print(b, w)

### **<font color="#39FF14" size=6 >重新定义损失函数</font>**

In [ ]:
lr = 0.1

# Step 0
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

optimizer = optim.SGD([b, w], lr=lr)

loss_fn = nn.MSELoss(reduction='mean')

n_epochs = 1000

for epoch in range(n_epochs):
    # Step 1
    yhat = b + w * x_train_tensor

    # Step 2
    loss = loss_fn(yhat, y_train_tensor)

    # Step 3
    loss.backward()

    # Step 4
    optimizer.step()
    optimizer.zero_grad()

print(b, w)

### **<font color="#39FF14" size=6 >定义模型</font>**

In [ ]:
class ManualLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.b = nn.Parameter(torch.randn(1,
                                          requires_grad=True,
                                          dtype=torch.float))
        self.w = nn.Parameter(torch.randn(1,
                                          requires_grad=True,
                                          dtype=torch.float))

    def forward(self, x):
        return self.b + self.w * x

In [ ]:
lr = 0.1

# Step 0
torch.manual_seed(42)

model = ManualLinearRegression().to(device)

optimizer = optim.SGD(model.parameters(), lr=lr)

loss_fn = nn.MSELoss(reduction='mean')

n_epochs = 1000

for epoch in range(n_epochs):
    model.train() # What is this?!?

    # Step 1
    yhat = model(x_train_tensor)

    # Step 2
    loss = loss_fn(yhat, y_train_tensor)

    # Step 3
    loss.backward()

    # Step 4
    optimizer.step()
    optimizer.zero_grad()

print(model.state_dict())

In [ ]:
class MyLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

In [ ]:
torch.manual_seed(42)
dummy = MyLinearRegression().to(device)
list(dummy.parameters())

In [ ]:
dummy.state_dict()

In [ ]:
torch.manual_seed(42)
# Building the model from the figure above
model = nn.Sequential(nn.Linear(1, 1)).to(device)

model.state_dict()

- Convolution Layers[卷积层](https://docs.pytorch.org/docs/stable/nn.html#convolution-layers)
- Pooling Layers[池化层](https://docs.pytorch.org/docs/stable/nn.html#pooling-layers)
- Padding Layers[填充层](https://docs.pytorch.org/docs/stable/nn.html#padding-layers)
- Non-linear Activations[非线性激活层](https://docs.pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity)
- Normalization Layers[归一化层](https://docs.pytorch.org/docs/stable/nn.html#normalization-layers)
- Recurrent Layers[循环层](https://docs.pytorch.org/docs/stable/nn.html#recurrent-layers)
- [Transformer Layers](https://docs.pytorch.org/docs/stable/nn.html#transformer-layers)
- Linear Layers[线性层](https://docs.pytorch.org/docs/stable/nn.html#linear-layers)
- Dropout Layers[丢弃层](https://docs.pytorch.org/docs/stable/nn.html#dropout-layers)
- Sparse Layers (embeddings)[稀疏层(嵌入式编码层)](https://docs.pytorch.org/docs/stable/nn.html#sparse-layers)
- Vision Layers[视觉层](https://docs.pytorch.org/docs/stable/nn.html#vision-layers)
- DataParallel Layers (multi-GPU)[并行数据层](https://docs.pytorch.org/docs/stable/nn.html#dataparallel-layers-multi-gpu-distributed)

In [ ]:
torch.manual_seed(42)
# Building the model from the figure above
model = nn.Sequential()
model.add_module('layer1', nn.Linear(3, 5))
model.add_module('layer2', nn.Linear(5, 1))
model.to(device)

### **<font color="#39FF14" size=6 >回顾：从Numpy到PyTorch</font>**

- 至此，我们已经使用PyTorch框架代替Numpy实现了简单线性回归的训练代码，代码包含3个基本组成：
- 数据准备
- 模型配置
    1. 模型
    2. 损失函数
    3. 优化器
- 模型训练
    1. 计算模型的预测值
    2. 计算损失
    3. 计算梯度
    4. 更新参数

### **<font color="#39FF14" size=6 >数据准备：Dataset和DataLoader</font>**

#### <font color="#CCFF00">**Dataset的基本结构**</font>

In [ ]:
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

# 自定义Dataset类
class CustomDataset(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor # 存储特征数据
        self.y = y_tensor # 存储标签数据
    
    def __getitem__(self, index):
        # 根据索引返回一个样本（特征，标签）
        return (self.x[index], self.y[index])
    
    def __len__(self):
        # 返回数据集的总样本数
        return len(self.x)

In [ ]:
# 构建tensor（注意：这里不发送到device，保持在CPU上）
x_train_tensor = torch.as_tensor(x_train).float()
y_train_tensor = torch.as_tensor(y_train).float()

# 使用自定义Dataset
train_data = CustomDataset(x_train_tensor, y_train_tensor)
print(f'Dataset大小: {len(train_data)}')
print(f'第一个数据点: {train_data[0]}')

#### <font color="#CCFF00">**使用TensorDataset简化**</font>

In [ ]:
# 使用TensorDataset（更简洁的方式）
train_data = TensorDataset(x_train_tensor, y_train_tensor)
print(f'Dataset大小: {len(train_data)}')
print(f'第一个数据点: {train_data[0]}')

#### <font color="#CCFF00">**DataLoader：实现小批量梯度下降**</font>

💡 <font color="#FF00A0">**关于batch size的选择**</font>：

通常使用2的幂次方（如16, 32, 64, 128），32是最常见的选择。batch size受硬件内存限制。

In [ ]:
# 创建DataLoader
train_loader = DataLoader(
    dataset=train_data,
    batch_size=16,  # 每个mini-batch包含16个数据点
    shuffle=True    # 每个epoch打乱数据顺序
)

# 查看一个mini-batch
x_batch, y_batch = next(iter(train_loader))
print(f'Features batch shape: {x_batch.shape}')
print(f'Labels batch shape: {y_batch.shape}')

#### <font color="#CCFF00">**使用random_split划分训练集和验证集**</font>

In [ ]:
torch.manual_seed(13)

# 先构建包含全部数据的tensor
x_tensor = torch.as_tensor(x).float()
y_tensor = torch.as_tensor(y).float()

# 构建完整数据集
full_dataset = TensorDataset(x_tensor, y_tensor)

# 划分训练集和验证集（80%训练，20%验证）
ratio = 0.8
n_total = len(full_dataset)
n_train = int(n_total * ratio)
n_val = n_total - n_train

train_data, val_data = random_split(full_dataset, [n_train, n_val])

print(f'训练集大小: {len(train_data)}')
print(f'验证集大小: {len(val_data)}')

In [ ]:
# 为训练集和验证集分别创建DataLoader
train_loader = DataLoader(
    dataset=train_data,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    dataset=val_data,
    batch_size=16,
    shuffle=False  # 验证集不需要打乱
)

print(f'训练batch数量: {len(train_loader)}')
print(f'验证batch数量: {len(val_loader)}')

#### <font color="#CCFF00">**为什么训练时要对数据进行shuffle，测试时不需要**</font>

#### <font color="#CCFF00">**重写训练循环以适应mini-batch**</font>

In [ ]:
# 定义训练步骤函数（与之前相同）
def make_train_step_fn(model, loss_fn, optimizer):
    def train_step(x, y):
        model.train()
        yhat = model(x)
        loss = loss_fn(yhat, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        return loss.item()
    return train_step

# 定义验证步骤函数
def make_val_step_fn(model, loss_fn):
    def val_step(x, y):
        model.eval()
        yhat = model(x)
        loss = loss_fn(yhat, y)
        return loss.item()
    return val_step

In [ ]:
# 辅助函数：处理mini-batch的内循环
def mini_batch(device, data_loader, step_fn):
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        mini_batch_loss = step_fn(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)
    return sum(mini_batch_losses) / len(mini_batch_losses)

In [ ]:
# 模型配置
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss(reduction='mean')

train_step_fn = make_train_step_fn(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)

# 训练循环
n_epochs = 200
losses = []
val_losses = []

for epoch in range(n_epochs):
    # 训练阶段
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)
    
    # 验证阶段（不计算梯度）
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)

print(f'最终训练损失: {losses[-1]:.4f}')
print(f'最终验证损失: {val_losses[-1]:.4f}')
print(f'模型参数: {model.state_dict()}')

### **<font color="#39FF14" size=6 >训练模式与验证模式</font>**

<font color="#FF00A0">**模式设置的重要性**</font>

### **<font color="#39FF14" size=6 >训练过程监控：TensorBoard</font>**

#### <font color="#CCFF00">**SummaryWriter基本用法**</font>

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# 创建SummaryWriter
# 日志将保存在 runs/simple_linear_regression 文件夹中
writer = SummaryWriter('runs/simple_linear_regression')

print('SummaryWriter已创建')

<font color="#FF00A0">**add_graph()**</font>：可视化模型结构

In [ ]:
# 获取一个mini-batch作为示例输入
x_dummy, y_dummy = next(iter(train_loader))

# 添加模型结构到TensorBoard
# 需要先将模型和数据发送到同一设备
writer.add_graph(model, x_dummy.to(device))

print('模型结构已添加到TensorBoard')

<font color="#FF00A0">**add_scalars()**</font>：记录多个标量值（如训练损失和验证损失）

In [ ]:
# 重新训练并记录到TensorBoard
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss(reduction='mean')

train_step_fn = make_train_step_fn(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)

n_epochs = 200

for epoch in range(n_epochs):
    # 训练阶段
    loss = mini_batch(device, train_loader, train_step_fn)
    
    # 验证阶段
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
    
    # 记录损失到TensorBoard
    writer.add_scalars(
        main_tag='loss',
        tag_scalar_dict={
            'training': loss,
            'validation': val_loss
        },
        global_step=epoch
    )

# 关闭writer
writer.close()

print('训练完成，数据已记录到TensorBoard')
print(f'最终模型参数: {model.state_dict()}')

#### <font color="#CCFF00">**使用Matplotlib绘制损失曲线**</font>

如果不想使用TensorBoard，也可以用Matplotlib绘制损失曲线：

In [ ]:
import matplotlib.pyplot as plt
# 重新训练并保存损失值
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss(reduction='mean')

train_step_fn = make_train_step_fn(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)

n_epochs = 200
losses = []
val_losses = []

for epoch in range(n_epochs):
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)
    
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)

# 绘制损失曲线
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(losses, label='Training Loss', color='#4dabf7', linewidth=2)
ax.plot(val_losses, label='Validation Loss', color='#ff6b6b', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training and Validation Loss Over Time', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### **<font color="#39FF14" size=6 >保存及加载模型</font>**

#### <font color="#CCFF00">**保存模型状态**</font>

模型的完整状态包括：
- <font color="#FF00A0">**model.state_dict()**</font>：模型参数
- <font color="#FF00A0">**optimizer.state_dict()**</font>：优化器状态（如动量）
- <font color="#FF00A0">**losses/val_losses**</font>：损失历史
- <font color="#FF00A0">**epoch**</font>：当前epoch数
- 其他需要保存的信息

In [ ]:
# 保存检查点
checkpoint = {
    'epoch': n_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': losses,
    'val_loss': val_losses
}

torch.save(checkpoint, 'model_checkpoint.pth')
print('模型检查点已保存到 model_checkpoint.pth')

#### <font color="#CCFF00">**加载模型以恢复训练**</font>

In [ ]:
# 模拟重新开始：创建新的未训练模型
torch.manual_seed(42)
new_model = nn.Sequential(nn.Linear(1, 1)).to(device)
new_optimizer = optim.SGD(new_model.parameters(), lr=0.1)

print('加载前模型参数:', new_model.state_dict())

In [ ]:
# 加载检查点
checkpoint = torch.load('model_checkpoint.pth')

new_model.load_state_dict(checkpoint['model_state_dict'])
new_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
saved_epoch = checkpoint['epoch']
saved_losses = checkpoint['loss']
saved_val_losses = checkpoint['val_loss']

# 恢复训练时，务必设置模型为训练模式
new_model.train()

print('加载后模型参数:', new_model.state_dict())
print(f'保存时的epoch: {saved_epoch}')
print(f'保存时的最终损失: {saved_losses[-1]:.4f}')

#### <font color="#CCFF00">**加载模型用于预测（部署）**</font>

In [ ]:
# 部署/预测时，只需要加载模型参数，不需要优化器
deploy_model = nn.Sequential(nn.Linear(1, 1)).to(device)

checkpoint = torch.load('model_checkpoint.pth')
deploy_model.load_state_dict(checkpoint['model_state_dict'])

# 部署/预测时，务必设置模型为评估模式
deploy_model.eval()

print('部署模型参数:', deploy_model.state_dict())

In [ ]:
# 使用模型进行预测
new_inputs = torch.tensor([[0.2], [0.34], [0.57], [0.8]]).to(device)

with torch.no_grad():
    predictions = deploy_model(new_inputs)

print('输入:', new_inputs.cpu().numpy().flatten())
print('预测输出:', predictions.cpu().numpy().flatten())

# 验证：手动计算
state = deploy_model.state_dict()
w = state['0.weight'].item()
b = state['0.bias'].item()
print(f'\n模型参数: y = {b:.4f} + {w:.4f} * x')
print('手动验证:', [b + w * x.item() for x in new_inputs])

#### <font color="#CCFF00">**完整训练流程总结**</font>

In [ ]:
# ============ 完整训练流程示例 ============

# 1. 数据准备
torch.manual_seed(13)
x_tensor = torch.as_tensor(x).float()
y_tensor = torch.as_tensor(y).float()
full_dataset = TensorDataset(x_tensor, y_tensor)

n_total = len(full_dataset)
n_train = int(n_total * 0.8)
n_val = n_total - n_train
train_data, val_data = random_split(full_dataset, [n_train, n_val])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)

# 2. 模型配置
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss(reduction='mean')

train_step_fn = make_train_step_fn(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)

# 创建TensorBoard writer
writer = SummaryWriter('runs/final_training')
x_dummy, _ = next(iter(train_loader))
writer.add_graph(model, x_dummy.to(device))

# 3. 模型训练
n_epochs = 200
losses = []
val_losses = []

for epoch in range(n_epochs):
    # 训练
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)
    
    # 验证
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)
    
    # 记录到TensorBoard
    writer.add_scalars('loss', {'training': loss, 'validation': val_loss}, epoch)

writer.close()

# 4. 保存模型
checkpoint = {
    'epoch': n_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': losses,
    'val_loss': val_losses
}
torch.save(checkpoint, 'final_model.pth')

print('训练完成！')
print(f'最终训练损失: {losses[-1]:.4f}')
print(f'最终验证损失: {val_losses[-1]:.4f}')
print(f'模型参数: {model.state_dict()}')

### <font color="#FF6B00" size=6 >**作业4：深度学习训练流程时间**</font>

**作业要求请见作业4中的“作业4要求.md”**

**作业提交方式：**
   - 请以<font color="#FF00A0">**“学号_hw4”**</font>文件夹提交作业至 Gitee 代码库中个人文件夹下
      - “学号_hw4”文件夹中应<font color="#FF00A0">**仅**</font>包含如下1个文件：
        1. 深度学习流程实践的实验报告“学号_hw4.pdf”
   - 作业截止时间：<font color="#FF00A0">**2026年4月2日00：00**</font>

## <font color="#FFEA00">本章小结</font>

### 学习建议

1. **课前预习**：阅读核心概念清单，标记不理解的部分，带着问题听课

2. **课堂笔记**：在此文件中补充讲解细节、个人理解和疑问

3. **课后复习**：
   - 整理本节核心概念清单，确保每个概念都能用自己的话解释
   - 完成课后思考题，必要时查阅参考资料
   - 动手运行和修改代码，加深理解

4. **作业完成**：
   - 作业3：认真阅读参考资料，理解自动微分原理（无需提交）
   - 作业4：按照要求完成深度学习训练流程实践，按时提交

5. **拓展阅读**：
   - PyTorch官方文档：https://pytorch.org/docs/stable/index.html
   - 《动手学深度学习》：https://zh.d2l.ai/

### 版权信息

---

**课程名称**：深度学习

**第二章**：神经网络训练流程

**版本**：学生分发版

**最后更新**：2026年3月

---

© 2026 深度学习讲习班 主讲教师版权所有

本课件仅供学生个人学习使用，未经许可不得用于商业用途或任何形式的传播。